## Projeto FLY

### Indicadores

### Gráficos

### Carteira e Rebalanceamento

### Análise de Portfolio

## Alchemy


In [ ]:
# Jupyter cell: Initialize and optimize SQLite in FLY

# 1) Imports
from sqlalchemy import create_engine, text
from sqlalchemy.orm import sessionmaker

from infrastructure.config import Config

In [ ]:
# 2) Load application configuration
#    Adjust constructor as needed (e.g., passing path to config file or env vars)
config = Config()

In [ ]:
# 3) Create the SQLAlchemy engine with thread-safe and future settings
engine = create_engine(
    config.database.connection_string,
    connect_args={"check_same_thread": False},  # allow multi-thread access
    future=True,
)

In [ ]:
# 4) Execute SQLite PRAGMAs for performance & integrity
with engine.connect() as conn:  # type: ignore
    conn.execute(text("PRAGMA wal_checkpoint"))  # type: ignore

    # ou, se preferir garantir que todo o WAL seja aplicado e truncado:
    conn.execute(text("PRAGMA wal_checkpoint(FULL)"))  # type: ignore
    conn.execute(text("PRAGMA wal_checkpoint(RESTART)"))  # type: ignore
    conn.execute(text("PRAGMA wal_checkpoint(TRUNCATE)"))  # type: ignore
    conn.execute(text("PRAGMA optimize"))  # type: ignore # Run internal optimizations
    conn.execute(text("PRAGMA journal_mode=WAL"))  # type: ignore # Write-Ahead Logging
    conn.execute(text("PRAGMA synchronous=FULL"))  # type: ignore # Full disk sync for safety
    conn.execute(text("PRAGMA foreign_keys=ON"))  # type: ignore # Enforce FK constraints
    conn.execute(text("PRAGMA temp_store=MEMORY"))  # type: ignore # RAM for temp tables
    conn.execute(text("PRAGMA cache_size=-65536"))  # type: ignore # 64 MB cache

In [ ]:
# 5) Create a session factory for ORM transactions
Session = sessionmaker(  # type: ignore
    bind=engine,  # type: ignore
    autoflush=True,
    expire_on_commit=True,
)

In [ ]:
# 6) (Optional) Inspect existing tables
with Session() as session:  # type: ignore
    result = session.execute(text("SELECT name FROM sqlite_master WHERE type='table';"))  # type: ignore
    tables = [row[0] for row in result]
    print("⏺️ Tables in database:", tables)

## YFINANCE

In [ ]:
import yfinance as yf

tickers = ["LREN3.SA", "VALE3.SA"]

# Histórico diário
df = yf.download(tickers, start="1960-01-01", interval="1d",)


In [ ]:
df

### Preço atual e variação intraday

In [ ]:
info = yf.Tickers(" ".join(tickers))

for t in tickers:
    data = info.tickers[t].info
    print(
        t,
        "Preço:", data.get("regularMarketPrice"),
        "Variação:", data.get("regularMarketChange"),
        "Variação %:", data.get("regularMarketChangePercent"),
    )


### Dados fundamentais

In [ ]:
data = {}

for t in tickers:
    tk = yf.Ticker(t)
    info = tk.info
    data[t] = {
        "Nome": info.get("longName"),
        "Setor": info.get("sector"),
        "Indústria": info.get("industry"),
        "Valor de Mercado": info.get("marketCap"),
        "P/L": info.get("trailingPE"),
        "Lucro por Ação (EPS)": info.get("trailingEps"),
        "Dividend Yield": info.get("dividendYield"),
        "Beta": info.get("beta"),
        "Moeda": info.get("currency"),
    }

import pandas as pd

df = pd.DataFrame(data).T
df


In [ ]:

t = yf.Ticker("VAle3.SA")

# Retorna todos os campos de fundamentos
info = t.info

# Converter para DataFrame para melhor leitura
df = pd.DataFrame(info.items(), columns=["Campo", "Valor"])
df

### Dados financeiros estruturados

In [ ]:
import yfinance as yf

tickers = ["LREN3.SA"]

for t in tickers:
    tk = yf.Ticker(t)

    print(f"\n--- {t} ---")

    # Demonstrativos de resultado (anual e trimestral)
    print("Income Statement (anual):")
    print(tk.financials)

    print("\nIncome Statement (trimestral):")
    print(tk.quarterly_financials)

    # Balanço patrimonial
    print("\nBalance Sheet (anual):")
    print(tk.balance_sheet)

    print("\nBalance Sheet (trimestral):")
    print(tk.quarterly_balance_sheet)

    # Fluxo de caixa
    print("\nCashflow (anual):")
    print(tk.cashflow)

    print("\nCashflow (trimestral):")
    print(tk.quarterly_cashflow)


In [ ]:
tk.eps_revisions

## BCB Series

### BCB Series Download

In [ ]:
import csv
import os
import re
import time

import requests
from bs4 import BeautifulSoup

In [ ]:
# --- FILE NAMES
csv_file_name = 'bcb_series_completas2.csv'
csv_headers = [
        'Codigo', 'Nome_Completo', 'Unidade', 'Periodicidade', 'Data_Inicio', 'Ultimo_Valor', 'Fonte'
    ]

# --- URLs de Ação (Sequência Mínima) ---
BASE_URL = "https://www3.bcb.gov.br/sgspub"
SEARCH_POST_URL = f"{BASE_URL}/localizarseries/localizarSeries.paint?method=localizarSeriesPorPesquisaAvancada"
PAGINATION_URL = f"{BASE_URL}/localizarseries/localizarSeries.do?method=getPagina"

METADATA_URL = f'{BASE_URL}/localizarseries/localizarSeries.do?method=recuperarMetadadosPorDocn'
METADADOS_BASICOS_URL = f'{BASE_URL}/JSP/consultarmetadados/cmiDadosBasicos.jsp'
METADADOS_COMPLETOS_URL = f'{BASE_URL}/JSP/consultarmetadados/cmiMetadados.jsp'

# --- HEADERS ---
HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36',
    'Content-Type': 'application/x-www-form-urlencoded',
}

# --- PAYLOADS ---
INITIAL_PAYLOAD = {
    'codigos': '', 'periodicidade': '0', 'fonte': '0', 'unidadePadrao': '0',
    'filtro': '0', 'chkClassificacao': 'on', 'txtPesquisa': '', 'ordem1': '0',
    'sentido1': '0', 'ordem2': '0', 'sentido2': '0', 'ordem3': '0',
    'sentido3': '0', 'hdCodigos': '', 'hdGrupos': '', 'hdTipoPesquisa': '9',
    'hdTipoOrdenacao': '0', 'hdPeriodicidade': 'Todas', 'hdFonte': 'Todas',
    'hdUnidadePadrao': 'Todas', 'hdFiltro': ''
}
BASE_PAYLOAD_PAGINATION = {
    'periodicidade': '0', 'codigo': '', 'fonte': '341', 'texto': '',
    'hdFiltro': '', 'hdOidGrupoSelecionado': '', 'hdSeqGrupoSelecionado': '',
    'hdNomeGrupoSelecionado': '', 'hdTipoPesquisa': '9', 'hdTipoOrdenacao': '0',
    'hdPeriodicidade': 'Todas', 'hdSeriesMarcadas': '', 'hdMarcarTodos': '',
    'hdFonte': '', 'hdOidSerieMetadados': '', 'hdNumeracao': '',
    'hdOidSeriesLocalizadas': '',
    'linkRetorno': '/sgspub/consultarvalores/telaCvsSelecionarSeries.paint',
    'linkCriarFiltros': '/sgspub/manterfiltros/telaMfsCriarFiltro.paint'
}
METADATA_PAYLOAD_BASE = { 
    # ... (Campos do seu payload de metadados aqui) ...
    'hdOidSerieMetadados': '', # OBRIGATÓRIO: Código da Série
    'linkRetorno': '/sgspub/consultarvalores/telaCvsSelecionarSeries.paint',
    'linkCriarFiltros': '/sgspub/manterfiltros/telaMfsCriarFiltro.paint'
}
HEADERS_TO_IGNORE = ["Sumário Metodológico", "Formatos de Divulgação", "Summary Methodology", "Dissemination Formats", ]


In [ ]:
def clean_cell(text):
    if not isinstance(text, str):
        return text

    text = text.replace('\x96', '–').replace('\x93', '"').replace('\x94', '"')
    text = text.replace('\xa0', ' ') # Remove Non-breaking space
    text = text.replace('*', '') # <--- REMOVE O ASTERISCO DE QUALQUER CHAVE OU VALOR

    # 2. Regex: Substitui qualquer sequência de whitespace por um único espaço
    text= re.sub(r'\s+', ' ', text).strip()

    return text


In [ ]:
def save_to_csv(data_list, csv_file_name):
    if not data_list:
        print("Nenhum dado para salvar em CSV.")
        return

    csv_headers = []

    # 2. Set para garantir a unicidade e rapidez na checagem
    seen_fields = set()

    # Itera sobre cada dicionário de metadados
    for series_dict in data_list:
        # Itera sobre as chaves do dicionário (a ordem de iteração é importante)
        for field in series_dict.keys():
            if field not in seen_fields:
                # Adiciona à lista de cabeçalhos e ao set de controle
                csv_headers.append(field)
                seen_fields.add(field)

    try:
        with open(csv_file_name, 'w', newline='', encoding='utf-8') as f:
            writer = csv.DictWriter(f, fieldnames=csv_headers, quoting=csv.QUOTE_ALL, restval='')

            writer.writeheader()
            writer.writerows(data_list)

        print(f"Total de {len(data_list)} séries salvas em '{csv_file_name}'.")

    except Exception as e:
        print(f"ERRO ao salvar o arquivo CSV: {e}")

In [ ]:
def parse_page_data(html_content):
    """Extrai os dados da tabela HTML, retornando uma lista de dicionários."""
    soup = BeautifulSoup(html_content, 'html.parser')

    table = soup.find('table', id='tabelaSeries')
    if not table:
        return []

    series_data = []

    # Busca todas as linhas (<tr>) diretamente na <table>.
    all_rows = table.find_all('tr')

    for row in all_rows:
        cells = row.find_all('td', recursive=False)

        # Ignora linhas de cabeçalho e linhas incompletas.
        if len(cells) < 10:
            continue

        # Mapeamento dos dados
        data = {
            'Codigo': clean_cell(cells[1].text),
            'Nome_Completo': clean_cell(cells[2].text),
            'Unidade': clean_cell(cells[3].text),
            'Periodicidade': clean_cell(cells[4].text),
            'Data_Inicio': clean_cell(cells[5].text),
            'Ultimo_Valor': clean_cell(cells[6].text),
            'Fonte': clean_cell(cells[7].text),
        }
        series_data.append(data)

    return series_data

In [ ]:
def fetch_series(series, soup_basico, metadata_completo):
    metadata = {}
    metadata_basico = soup_basico.find_all('table')
    for t, table in enumerate(metadata_basico):
        rows = table.find_all('tr')
        for r, row in enumerate(rows):
            # Buscar células de dados (<td>)
            cells = row.find_all('td')
            # Assume a estrutura de 4 colunas (Rótulo, Valor, Rótulo, Valor)
            for i in range(0, len(cells), 2):
                if i + 1 < len(cells):
                    label = clean_cell(cells[i].text)
                    value = clean_cell(cells[i+1].text)
                    if label:
                        metadata[label] = value

    last_cell = None
    for t, table in enumerate(metadata_completo):
        rows = table.find_all('tr')
        for r, row in enumerate(rows):
            cells = row.find_all('td')
            cell_count = len(cells)
            if cell_count > 2:
                continue
            elif cell_count == 2:
                # Condição: Rótulo e Valor na mesma linha (Estrutura 2-col)
                # O bloco anterior de iteração foi removido
                label = clean_cell(cells[0].text)
                value = clean_cell(cells[1].text)

                if label and label not in HEADERS_TO_IGNORE:
                    metadata[label] = value

                # Zera o last_cell, pois um par K:V completo foi encontrado
                last_cell = None 

            elif cell_count == 1:
                # Condição: Máquina de Estados para conteúdo multi-linha (Estrutura 1-col)
                content = clean_cell(cells[0].text)

                # 1. Se existe uma chave pendente (last_cell):
                if last_cell:
                    # O 'content' atual é o VALOR para a chave pendente.
                    # Se o conteúdo estiver vazio, armazena vazio. Se tiver texto longo, armazena o texto limpo.
                    metadata[last_cell] = content
                    # Zera o estado após encontrar o valor
                    last_cell = None 

                # 2. Se NÃO existe chave pendente e o 'content' é um rótulo válido:
                elif content and content not in HEADERS_TO_IGNORE:
                    # O 'content' é um novo rótulo (chave) que espera um valor na próxima linha
                    last_cell = content
    series.update(metadata)

    return series


In [ ]:
def scrape_all_series(page_num=1):
    """Executa o scraping completo e retorna a lista de dicionários de séries."""
    all_data = []
    session = requests.Session()
    max_retries = 3

    # --- 1. ESTABELECIMENTO DE SESSÃO ---
    response_session = session.get(BASE_URL)
    response_initial = session.post(SEARCH_POST_URL, data=INITIAL_PAYLOAD, headers=HEADERS, timeout=30)

    # --- 2. LOOP DE PAGINAÇÃO ---
    page_num = page_num or 1
    previous_content_hash = hash(response_initial.text)
    previous_content_hash = hash(response_session.text)

    while True:
        current_payload = BASE_PAYLOAD_PAGINATION.copy()
        current_payload['hdNumPagina'] = str(page_num)

        retries = 0
        current_html = ""

        # Tenta o request de paginação
        while retries < max_retries:
            try:
                response_page = session.post(PAGINATION_URL, data=current_payload, headers=HEADERS, timeout=10)
                response_page.raise_for_status() 
                current_html = response_page.text
                break
            except requests.exceptions.RequestException:
                retries += 1
                time.sleep(3)

        if retries == max_retries:
            print(f"Falha na página {page_num}. Abortando.")
            break

        # Verifica se o conteúdo é o mesmo da página anterior (indicador de fim)
        current_content_hash = hash(current_html.replace('\r\n', ''))
        if current_content_hash == previous_content_hash:
            print(f"Conteúdo da página {page_num} idêntico. Fim da paginação.")
            break

        previous_content_hash = current_content_hash

        data_page = parse_page_data(current_html)

        if not data_page:
            print(f"Página {page_num} não retornou linhas. Encerrando.")
            break

        series_data = []
        for s, series in enumerate(data_page):
            payload = METADATA_PAYLOAD_BASE.copy()
            codigo = series['Codigo'] # Inserindo o código dinamicamente
            fonte = series['Fonte']
            nome = series['Nome_Completo']
            payload['hdOidSerieMetadados'] = codigo

            print(f"Página {page_num}-{(len(all_data)+s)} {codigo}-{fonte}-{nome}")
            session.post(METADATA_URL, data=payload, headers=HEADERS, timeout=10)
            response_basico = session.get(METADADOS_BASICOS_URL, headers=HEADERS, timeout=10)
            soup_basico = BeautifulSoup(response_basico.text, 'html.parser')

            response_completo = session.get(METADADOS_COMPLETOS_URL, headers=HEADERS, timeout=10)
            soup_completo = BeautifulSoup(response_completo.text, 'html.parser')
            metadata_completo = soup_completo.find_all('table')

            updated_series = fetch_series(series, soup_basico, metadata_completo)
            series_data.append(updated_series) # Usa append() para adicionar o dicionário)

        all_data.extend(series_data)
        if all_data is not None:
            save_to_csv(all_data, csv_file_name)
            print(f"Página {page_num} capturada. Total geral: {len(all_data)}")
        else:
            print("Processo de scraping falhou. Verifique as mensagens de erro acima.")

        page_num += 1
        # time.sleep(0.3)

    print("\nColeta finalizada.")
    return all_data

In [ ]:
# 1. Executa o scraping e armazena o resultado
bcb_series = scrape_all_series()

# 2. Salva a lista em CSV
if bcb_series is not None:
    save_to_csv(bcb_series, csv_file_name)
else:
    print("Processo de scraping falhou. Verifique as mensagens de erro acima.")

In [ ]:
df = pd.read_csv(r"D:\Fausto Stangler\Documentos\Python\FLY\bcb_series_completas.csv", dtype=str, )


In [ ]:
fontes = df['Fonte'].unique()
fontes

In [ ]:
dfs = {}
for fonte in fontes:
    mask = df['Fonte'] == fonte
    dfs[fonte] = df[mask][['Codigo', 'Short name', ]].drop_duplicates()

In [ ]:
df['Fonte'] == 'FGV'

In [ ]:
mask = df['Fonte'] == 'FGV'
df[mask]

### BCB Series CSV

In [ ]:
import os
import re

import pandas as pd


In [ ]:
csv_file = r"D:\Fausto Stangler\Documentos\Python\FLY\bcb_series_completas.csv"
df = pd.read_csv(csv_file, dtype=str, low_memory=False)
df['Codigo'] = df['Codigo'].astype('Int64')
cols = ['Codigo', 'Fonte', 'Short name', 'Full name', 'Periodicity', 'Ultimo_Valor', 'Series type', 'Subject chain']
df = df[cols]
df

In [ ]:
def parse_date(x):
    x = str(x).strip().lower()
    if re.match(r'^\d{4}$', x):  # ano puro
        return pd.Timestamp(f"{x}-12-31")
    if re.match(r'^[a-z]{3}/\d{4}$', x):  # formato tipo jan/2024
        return pd.to_datetime(x, format='%b/%Y', errors='coerce')
    if re.match(r'^\d{2}/\d{2}/\d{4}$', x):  # formato tipo 31/12/2024
        return pd.to_datetime(x, format='%d/%m/%Y', errors='coerce')
    if "quar" in x or "tri" in x:  # trimestre (ex: '1º Quar. 2024')
        ano = re.findall(r'\d{4}', x)
        mes = re.findall(r'1º|2º|3º|4º', x)
        if ano and mes:
            trimestre = {"1º": "03", "2º": "06", "3º": "09", "4º": "12"}[mes[0]]
            return pd.Timestamp(f"{ano[0]}-{trimestre}-30")
    return pd.NaT

df["Ultimo_Valor_dt"] = df["Ultimo_Valor"].apply(parse_date)
df = df[df["Ultimo_Valor_dt"] >= pd.Timestamp("2024-01-01")]
df

In [ ]:
subject_chain = df['Subject chain'][df['Subject chain'].notna()].unique()

processed_chain = [
    [segment.strip() for segment in item.split('-') if segment.strip()]
    for item in subject_chain
]
processed_chain = pd.DataFrame(processed_chain)



In [ ]:
subjects = []
for idx, row in processed_chain.iterrows():
    chain = []
    for col in processed_chain.columns:
        val = row[col]
        if pd.isna(val):
            break
        chain.append(val)
    subjects.append(chain)
len(subjects)


In [ ]:
masks = {}
for chain in subjects:
    pattern = '.*'.join(map(re.escape, chain))
    index = "-".join(chain)
    mask = df['Subject chain'].str.contains(pattern, case=False, na=False)
    subset = df[mask]
    masks[index] = subset



In [ ]:
folder = "bcb_series"
os.makedirs(folder, exist_ok=True)

for index, subset in masks.items():
    safe_name = re.sub(r'[\\/*?:"<>|]', "_", index)
    path = os.path.join(folder, f"{safe_name}.csv")
    subset.to_csv(path, index=False)


In [ ]:
xlsx_path = os.path.join(folder, "bcb_series.xlsx")

with pd.ExcelWriter(xlsx_path, engine="xlsxwriter") as writer:
    for index, subset in masks.items():
        sheet_name = re.sub(r'[^A-Za-z0-9]', "_", index)[:31]  # Excel limita a 31 caracteres
        subset.to_excel(writer, sheet_name=sheet_name, index=False)


In [ ]:
combined = pd.concat(
    [subset.assign(index_name=index) for index, subset in masks.items()],
    ignore_index=True
)
path = os.path.join(folder, f"index.csv")
combined.to_csv(path)

## Indicators

In [ ]:
import os
import sqlite3
from pathlib import Path

import matplotlib
import pandas as pd


In [ ]:
DB_PATH = Path(r"D:\Fausto Stangler\Documentos\Python\FLY\data\fly.db")
TABLE_NAME = "tbl_indicators"


In [ ]:
# 1. Conexão ao banco de dados SQLite
conn = sqlite3.connect(DB_PATH)

# 2. Leitura da tabela para o DataFrame 'df'
query = f"SELECT * FROM {TABLE_NAME}"
df = pd.read_sql_query(query, conn)

# 3. Fechamento da conexão
conn.close()


In [ ]:
df

In [ ]:
items = df[['name','code']].drop_duplicates()

In [ ]:
codes = {}
for item in items.iterrows():
    code = item[1].iloc[1]
    name = item[1].iloc[0]
    codes[code] = name
    print(code, name)

In [ ]:
code = '433'
mask = df['name'] == codes[str(code)]
mask&= df['date'] > '1995-01-01'
df[mask][['date','value']].set_index('date').plot()
print(codes[str(code)])

## SQLITE


In [ ]:
import sqlite3

import matplotlib
import pandas as pd

In [ ]:
db_path = r"D:\Fausto Stangler\Documentos\Python\FLY\data\fly.db"
raw = "tbl_statements_raw"
fetched = "tbl_statements_fetched"
stock_quote = "tbl_stock_quote"

In [ ]:
# conecta ao banco
conn = sqlite3.connect(db_path)

# lê a tabela em um DataFrame
df = pd.read_sql(f"SELECT * FROM {stock_quote}", conn)


In [ ]:
ticker = "ALPA3"

mask = df['ticker'] == ticker
df[mask]['close'].plot()

In [ ]:
# fecha a conexão se não precisar mais
conn.close()


## Ratios

### Alinhamentos

In [ ]:
import numpy as np
import pandas as pd

df_data = pd.read_csv("df_data.csv")
df_data['date'] = pd.to_datetime(df_data['date'], errors='coerce')
df_data.set_index('date', inplace=True)
df_data.index = pd.DatetimeIndex(df_data.index)

df_anchor_calendar = pd.read_csv("df_anchor_calendar.csv")
df_anchor_calendar['date'] = pd.to_datetime(df_anchor_calendar['date'], errors='coerce')
df_anchor_calendar.set_index('date', inplace=True)
df_anchor_calendar.index = pd.DatetimeIndex(df_anchor_calendar.index)


In [ ]:
df_data

In [ ]:
df_anchor_calendar

In [ ]:
cols = df_data.columns
numeric_cols = df_data.select_dtypes(include='number').columns
object_cols = df_data.select_dtypes(include='object').columns

# Define agregação por tipo
agg_dict = {}

# Object (strings, etc.) → last
for col in object_cols:
    agg_dict[col] = 'last'

# Numéricas → mean
for col in numeric_cols:
    agg_dict[col] = 'mean'
agg_dict['id'] = 'count'

In [ ]:
df_data2 = df_data.reset_index(drop=False)
df_data2['date'] = pd.to_datetime(df_data2['date'], errors='coerce')
df_data2 = df_data2.assign(_period=df_data2['date'].dt.to_period("M"))
df_data2

In [ ]:
df_data2['typical_price'] = (df_data2['high'] + df_data2['low'] + df_data2['close']) / 3
df_data2['num'] = df_data2['typical_price'] * df_data2['volume'].cumsum()
df_data2['denom'] = df_data2['volume'].cumsum()
df_data2['wvap'] = (df_data2['num'] / df_data2['denom'])
df_data2

In [ ]:
group = df_data2.groupby('_period')

monthly = group.agg({
    'open': 'first',
    'high': 'max',
    'low': 'min',
    'close': 'last',
    'adj_close': 'last',
    'volume': 'sum',
    'ticker': 'last',
    'company_name': 'last',
    'id': 'count',
    'currency': 'last',
})


numer = (df_data2['typical_price'] * df_data2['volume']).groupby(df_data2['_period']).sum()
denom = df_data2['volume'].groupby(df_data2['_period']).sum()
monthly['vwap'] = np.where(denom > 0, numer / denom, np.nan)
monthly['desviation_vwap %'] = (monthly['typical_price'] - monthly['vwap']) / monthly['typical_price'] * 100

stats = group['vwap'].agg(['mean', 'std'])
monthly = monthly.join(stats, on='_period')




monthly

In [ ]:
#. agrega por mês usando PeriodIndex para evitar depender de datas de fim-de-mês
df_data.index = pd.DatetimeIndex(df_data.index)

monthly_data = (
    df_data
    .assign(_period=df_data.index.to_period('M'))
    .groupby('_period')
    .agg(agg_dict)
)
monthly_data

In [ ]:
cols = df_anchor_calendar.columns
numeric_cols = df_anchor_calendar.select_dtypes(include='number').columns
object_cols = df_anchor_calendar.select_dtypes(include='object').columns

# Define agregação por tipo
agg_dict = {}

# Object (strings, etc.) → last
for col in object_cols:
    agg_dict[col] = 'first'

# Numéricas → mean
for col in numeric_cols:
    agg_dict[col] = 'sum'

In [ ]:
monthly_data = (
    df_anchor_calendar
    .assign(_period=df_anchor_calendar.index.to_period('M'))
    .groupby('_period')
    .agg(agg_dict)
)
monthly_data[cols]

In [ ]:
# Executa
monthly = df_data.groupby(pd.Grouper(freq='B')).agg(agg_dict)
dropped = monthly[monthly[numeric_cols].isna().any(axis=1)]
monthly[cols]

In [ ]:
import pandas as pd

# df_anchor_calendar: calendário MENSAL (baixa granularidade)
anchor_index = pd.to_datetime(['2024-01-31', '2024-02-29', '2024-03-31'])
df_anchor_calendar = pd.DataFrame({'month': ['Jan', 'Fev', 'Mar']}, index=anchor_index)

# df_data: cotações DIÁRIAS (alta granularidade)
data_index = pd.to_datetime([
    '2024-01-15', '2024-01-16', '2024-01-30', '2024-01-31',
    '2024-02-15', '2024-02-29',
    '2024-03-31', '2024-04-01'  # fora do range!
])
df_data = pd.DataFrame({
    'close': [100, 101, 102, 103, 104, 105, 106, 107]
}, index=data_index)

print("df_data (alta granularidade):")
print(df_data)

In [ ]:
# REINDEX para o calendário mensal
df_aligned = df_data.reindex(df_anchor_calendar.index, method='ffill')
print("\ndf_aligned (após reindex):")
print(df_aligned)

In [ ]:
# 1. Agrupe df_data por mês (mantendo alta granularidade)
monthly_raw = df_data.groupby(df_data.index.to_period('M')).agg({'close': 'mean'})

# 2. Reindex para o calendário final
monthly = monthly_raw.reindex(df_anchor_calendar.index.to_period('M')).to_timestamp(how='end')
monthly

### RAW code

In [ ]:
import sqlite3

import pandas as pd

In [ ]:
db_path  = r"D:\Fausto Stangler\Documentos\Python\FLY\data\fly.db"
conn = sqlite3.connect(db_path)


In [ ]:
df = pd.read_sql_query("SELECT * FROM tbl_statements_ratio", conn)
df['date'] = pd.to_datetime(df['date'])
df = df.set_index('date').sort_index()



In [ ]:
mask = df['ticker'] == 'AFLT3'
mask&= df['account'].str.startswith('99.3.close')

cols = ['value']

In [ ]:
df['company_name'].unique()

In [ ]:
df[mask][cols].plot()
df[mask][cols].drop_duplicates().plot()

### Cache

In [ ]:
import hashlib
import inspect
import json
import sqlite3
from pathlib import Path

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq


In [ ]:
def set_df_hash(df, row_hash="row_hash"):
    df = df.sort_index()
    df['date'] = df.index.strftime('%Y-%m-%d %H:%M:%S.%f')
    df[row_hash] = df.astype(str).agg('|'.join, axis=1).apply(lambda r: hashlib.sha256(r.encode("utf-8")).hexdigest())
    df = df.drop(columns=['date'])
    hash_df = hashlib.sha256("".join(df.sort_index()[row_hash]).encode("utf-8")).hexdigest()
    return df, hash_df


In [ ]:
def hash_diff(df_old, df_new, hash_col='row_hash'):
    if df_old.columns.tolist() != df_new.columns.tolist():
        return pd.DataFrame()
    hash_diff = df_old[[hash_col]].join(df_new[[hash_col]], lsuffix='_old', rsuffix='_new')
    hash_diff['changed'] = hash_diff[f'{hash_col}_old'] != hash_diff[f'{hash_col}_new']
    indices = hash_diff[hash_diff['changed']].index
    
    if len(indices) == 0:
        return pd.DataFrame()
    
    old = df_old.loc[indices]
    new = df_new.loc[indices]
    cols = [c for c in df_old.columns if c != hash_col]
    
    diffs = []
    for idx in indices:
        o = old.loc[idx, cols]
        n = new.loc[idx, cols]
        chg = o[o != n].index
        for col in chg:
            diffs.append({'data': idx, 'column': col, 'old': o[col], 'new': n[col]})
    
    return pd.DataFrame(diffs).sort_values(['data', 'column']).reset_index(drop=True)


In [ ]:
BASE = Path(".")
CACHE_DIR = BASE / "cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
SQLITE = BASE / "cache.db"

logical_name = "statements_ratio"
version = 1 # Increment this when the calculation logic (e.g., the ratio formula) changes


###### DataFrames

In [ ]:
indicators = pd.read_csv('indicators.csv')
indicators["date"] = pd.to_datetime(indicators["date"], errors="coerce")
indicators = indicators.set_index("date").sort_index()
i = indicators.sort_index().index

quotes = pd.read_csv("quotes.csv")
quotes["date"] = pd.to_datetime(quotes["date"], errors="coerce")
quotes = quotes.set_index("date").sort_index()
q = quotes.sort_index()

statements = pd.read_csv("statements.csv")
s = statements.sort_index().copy()

key_columns = ["company_name", "quarter", "account"]
# sep = " - "
s["account_description"] = s["account"] + " - " + s["description"] + " - " + s["grupo"] + " - " + s["quadro"]

context_columns = ["nsd", "company_name", "version"]
meta = (
    s[["quarter"] + context_columns]
    .drop_duplicates(subset=["quarter"], keep="last")
    .set_index("quarter")
)

s = s.sort_values(key_columns)
s = s.drop_duplicates(subset=key_columns, keep="last")
s = s.pivot_table(index=["quarter"],
                        columns="account_description",  # use "description" se preferir nomes ou "account" se preferir contas
                        values="value",
                        aggfunc="last").sort_index()
s.columns.name = None

s = meta.join(s, how="right")

if q is not None:
    # reset_multiindex
    s = s.copy()
    # s.index = s.index.set_names(["quarter", "nsd", "company_name", "version"])
    s = s.reset_index()

    # create date index
    s["quarter"] = pd.to_datetime(s["quarter"])
    s = (
        s.rename(columns={"quarter": "date"})
        .set_index("date")
        .sort_index()
    )

    # reindex ffill bfill
    s = s.reindex(q.index, method="ffill")
    if s.iloc[0].isna().any():
        s = s.bfill()

    # recreate multiindex
    s = s.set_index(context_columns, append=True).sort_index()
    s = s.reset_index(level=context_columns)
    quotes.to_csv('q.csv', index=True)
    s.to_csv('s.csv', index=True)

i_hash = hashlib.sha256(pd.util.hash_pandas_object(i, index=True).sum()).hexdigest()
q_hash = hashlib.sha256(pd.util.hash_pandas_object(q, index=True).sum()).hexdigest()
s_hash = hashlib.sha256(pd.util.hash_pandas_object(s, index=True).sum()).hexdigest()
i_hash, q_hash, s_hash

In [ ]:
q0, q0_hash = set_df_hash(q)
s0, s0_hash = set_df_hash(s)

s0_copy = s.copy()
s0_copy.loc['2006-11-21 00:00:00', '00.01.02 - Ações PN Circulação - Dados da Empresa - Composição do Capital'] = 1000

s1, s1_hash = set_df_hash(s0_copy)

q0_hash, s0_hash, s1_hash

In [ ]:
diffs = None
if s0_hash != s1_hash:
    diffs = hash_diff(s0, s1)
diffs

In [ ]:
def calculate_ratios(q, s):
    # CÓDIGO CRÍTICO DE CÁLCULO
    s = s.reindex(q.index, method='ffill') 
    # s_col = s.columns[[c.startswith("03.01") for c in s.columns.tolist()]]
    col_03_01 = '03.01 - Receita de Venda de Bens e/ou Serviços - DFs Individuais - Demonstração do Resultado'
    col_03_11 = '03.11 - Lucro do Período - DFs Individuais - Demonstração do Resultado'

    ratio_df = pd.DataFrame(index=q.index)

    ratio_df['PriceToRevenue'] = q['adj_close'] / s[col_03_01]
    ratio_df['PriceToEarnings'] = q['adj_close'] / s[col_03_11]
    ratio_df['ROE'] = s[col_03_11]  / s[col_03_01] * 100
    return ratio_df

# 1. Obter o código-fonte (a string) da função
calculate_ratios_str = inspect.getsource(calculate_ratios)
ratios_method_hash = hashlib.sha256(calculate_ratios_str.encode()).hexdigest()
ratios_method_hash

In [ ]:
app_hash = hashlib.sha256(f"{logical_name}|{version}".encode()).hexdigest()
cache_key = hashlib.sha256( f"{app_hash}{q_hash}{s_hash}{i_hash}{ratios_method_hash}".encode()).hexdigest()
cache_key

In [ ]:
import sqlite3
from datetime import datetime

DB_PATH = 'cache/fly_cache.db'

def init_cache_table():
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS cache (
            cache_key TEXT PRIMARY KEY,
            file_path TEXT NOT NULL,
            size_bytes INTEGER NOT NULL,
            created_at TEXT NOT NULL,
            accessed_at TEXT NOT NULL,
            access_count INTEGER DEFAULT 0,
            code_hash TEXT NOT NULL  -- NOVO: hash do método
        )
    ''')
    
    # Adicionar coluna se não existir (para compatibilidade)
    try:
        cursor.execute("ALTER TABLE cache ADD COLUMN code_hash TEXT")
    except sqlite3.OperationalError:
        pass  # Já existe
    
    conn.commit()
    conn.close()



In [ ]:
MAX_CACHE_SIZE_BYTES = 1_000_000_000
MAX_AGE_DAYS = 30  # Expira se não acessado em 30 dias

def evict_cache_if_needed(conn):
    cursor = conn.cursor()
    
    # 1. Expiração por tempo: Deletar velhos
    cutoff_time = (datetime.now() - pd.Timedelta(days=MAX_AGE_DAYS)).isoformat()
    cursor.execute("SELECT file_path FROM cache WHERE accessed_at < ?", (cutoff_time,))
    for (path,) in cursor.fetchall():
        if os.path.exists(path):
            os.remove(path)
    cursor.execute("DELETE FROM cache WHERE accessed_at < ?", (cutoff_time,))
    conn.commit()
    
    # 2. Expiração por tamanho: Enquanto total > MAX, deletar o menos acessado/mais antigo
    cursor.execute("SELECT SUM(size_bytes) FROM cache")
    total_size = cursor.fetchone()[0] or 0
    
    while total_size > MAX_CACHE_SIZE_BYTES:
        # Encontrar o a evictar: Ordenar por access_count ASC, created_at ASC (LFU + LRU simples)
        cursor.execute("""
            SELECT cache_key, file_path, size_bytes 
            FROM cache 
            ORDER BY access_count ASC, created_at ASC 
            LIMIT 1
        """)
        row = cursor.fetchone()
        if not row:
            break
        
        cache_key, path, size = row
        if os.path.exists(path):
            os.remove(path)
        cursor.execute("DELETE FROM cache WHERE cache_key = ?", (cache_key,))
        conn.commit()
        
        total_size -= size  # Atualizar total (aproximado; para precisão, requery)

In [ ]:
def invalidate_old_caches(code_hash=ratios_method_hash, conn=None):
    """
    Deleta todos os caches cujo code_hash != current_code_hash
    """
    evict_cache_if_needed(conn)  # Função abaixo

    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    
    # Pegar todos com code_hash diferente
    cursor.execute("""
        SELECT file_path, code_hash 
        FROM cache 
        WHERE code_hash != ?
    """, (code_hash,))
    
    deleted_count = 0
    for file_path, old_hash in cursor.fetchall():
        if os.path.exists(file_path):
            os.remove(file_path)
            print(f"[INVALIDATED] {file_path} (método antigo) {old_hash}")
        deleted_count += 1
    
    # Deletar do DB
    cursor.execute("DELETE FROM cache WHERE code_hash != ?", (code_hash,))
    conn.commit()
    conn.close()
    
    if deleted_count > 0:
        print(f"Invalidated {deleted_count} cache(s) due to method change.")
    else:
        print("All caches up to date.")

In [ ]:
import os

import pandas as pd

CACHE_DIR = 'cache/'  # Pasta para .parquets
os.makedirs(CACHE_DIR, exist_ok=True)

def get_or_compute_ratios(q, s, i=None, code_hash=ratios_method_hash):  # Assuma 'i' se for outro input
    init_cache_table()  # Rode uma vez ou sempre no startup
    # Calcular hashs (se não pré-computados)
    q_hash = hashlib.sha256(pd.util.hash_pandas_object(q, index=True).sum()).hexdigest()
    s_hash = hashlib.sha256(pd.util.hash_pandas_object(s, index=True).sum()).hexdigest()
    i_hash = hashlib.sha256(pd.util.hash_pandas_object(i, index=True).sum()).hexdigest() if i is not None else ''
    ratios_method_hash = hashlib.sha256(inspect.getsource(calculate_ratios).encode()).hexdigest()
    
    app_hash = hashlib.sha256(f"{logical_name}|{version}".encode()).hexdigest()
    cache_key = hashlib.sha256(f"{app_hash}{q_hash}{s_hash}{i_hash}{ratios_method_hash}".encode()).hexdigest()
    
    file_path = os.path.join(CACHE_DIR, f"{cache_key}")
    
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    
    try:# Verificar se existe
        cursor.execute("SELECT file_path, size_bytes FROM cache WHERE cache_key = ?", (cache_key,))
        row = cursor.fetchone()
        
        if row:
            db_file_path = row[0]
            if os.path.exists(row[0]):
                try:
                    print(f"Cache HIT: {cache_key}")
                    df = pd.read_parquet(file_path)
                    
                    # Atualizar accessed_at e access_count
                    cursor.execute("""
                        UPDATE cache 
                        SET accessed_at = ?, access_count = access_count + 1 
                        WHERE cache_key = ?
                    """, (datetime.now().isoformat(), cache_key))
                    conn.commit()
                    return df
                except Exception as e:
                    if os.path.exists(db_file_path):
                        os.remove(db_file_path)
                    cursor.execute("DELETE FROM cache WHERE cache_key = ?", (cache_key,))
                    conn.commit()
    
        print(f"Cache MISS: {cache_key}")
        
        # Eviction antes de adicionar novo (para controlar tamanho)
        invalidate_old_caches(code_hash=ratios_method_hash, conn=conn)

        # Computar
        df_ratios = calculate_ratios(q, s)  # Sua função (adicione 'i' se necessário)
        
        # Salvar .parquet
        df_ratios.to_parquet(file_path, compression='zstd')  # Compacto
        
        # Calcular tamanho
        size_bytes = os.path.getsize(file_path)
        
        now_iso = datetime.now().isoformat()
        
        # Inserir no DB
        cursor.execute('''
                INSERT INTO cache 
                (cache_key, file_path, size_bytes, created_at, accessed_at, access_count, code_hash)
                VALUES (?, ?, ?, ?, ?, ?, ?)
            ''', (cache_key, file_path, size_bytes, now_iso, now_iso, 1, code_hash))

        conn.commit()

    finally:
        conn.close()  # SEMPRE fecha    conn.close()

    return df_ratios

In [ ]:
df_ratios = get_or_compute_ratios(q, s, i)
df_ratios = get_or_compute_ratios(q, s0, q)
df_ratios = get_or_compute_ratios(q, s0, s)
df_ratios = get_or_compute_ratios(q, s0, s1)
df_ratios = get_or_compute_ratios(q, s0, q)
df_ratios = get_or_compute_ratios(q, s0, i)

df_ratios

In [ ]:
!pip install matplotlib

In [ ]:
import pandas as pd
import matplotlib

In [ ]:
filepath = r"D:\Fausto Stangler\Documentos\Python\FLY\data\cache\AMBEV-SA\15066d1996a533cdb2d6bae65f41f7bb19da09c7f7ff914224b1ebec9c8fe5f9.parquet"
df = pd.read_parquet(filepath)
# df['30.02 - Valor de Mercado Adicionado (MVA)'].plot()
df.to_csv('cache.csv')

df['02.03 - Patrimônio Líquido - DFs Consolidadas - Balanço Patrimonial Passivo'].plot()
df['02.03 - Patrimônio Líquido - DFs Consolidadas - Balanço Patrimonial Passivo']

In [ ]:
print("Colunas disponíveis:")
for c in sorted(df.columns):
    print(c)


In [ ]:
df['99.3.close - stock_3'].plot()

In [ ]:
df['99.3.close - stock_3'].plot()

In [ ]:
r = calculate_ratios(q, s)
r

In [ ]:
base = s.sort_index().copy()
base['date'] = base.index.map(lambda x: x.isoformat())
base['row_hash'] = base[sorted(base.columns.tolist())].astype(str).agg('|'.join, axis=1).apply(lambda x: hashlib.sha256(x.encode("utf-8")).hexdigest())
base = base.drop(columns=['date'])
hash_s = hashlib.sha256("".join(base.sort_index(ascending=True)["row_hash"]).encode("utf-8")).hexdigest()
hash_s

In [ ]:
snapshot_antigo = base[["row_hash"]].copy()
s1 = s.sort_index().copy()
s1.loc['2006-07-31 00:00:00', '00.01.02 - Ações PN Circulação - Dados da Empresa - Composição do Capital'] = 999
s1['date'] = s1.index.map(lambda x: x.isoformat())
s1['row_hash'] = s1[sorted(s1.columns.tolist())].astype(str).agg('|'.join, axis=1).apply(lambda x: hashlib.sha256(x.encode("utf-8")).hexdigest())
s1 = s1.drop(columns=['date'])
hash_s1 = hashlib.sha256("".join(s1.sort_index(ascending=True)["row_hash"]).encode("utf-8")).hexdigest()
hash_s1


In [ ]:
diff = hash_diff(base, s1)
diff

#### Code

In [ ]:
def hash_dataframe(df):
    h = hashlib.sha256()
    h.update(pd.util.hash_pandas_object(df, index=True).values)
    return h.hexdigest()


In [ ]:
cache_dir=Path("cache")
tolerance="1D"

In [ ]:
hash_dataframe(statements)

In [ ]:
key = hashlib.sha256(
    (hash_dataframe(quotes) + hash_dataframe(statements) + tolerance).encode()
).hexdigest()
path = cache_dir / f"{key}.parquet"
if path.exists():
    print("→ Lendo do cache:", path)
merged = pd.merge_asof(
    quotes.sort_index(), statements.sort_index(),
    left_index=True, right_index=True,
    direction="backward", tolerance=pd.to_timedelta(tolerance)
    )
merged = pd.merge_asof(
    quotes.sort_index(), statements.sort_index(),
    left_index=True, right_index=True,
    direction="backward", tolerance=None
)
merged.to_csv("merged.csv", index=False)


In [ ]:
df = statements.reindex(quotes.sort_index().index, method="ffill")
df

In [ ]:
cache_dir.mkdir(exist_ok=True)
key = hashlib.sha256(
    (hash_dataframe(quotes) + hash_dataframe(statements) + tolerance).encode()
).hexdigest()
path = cache_dir / f"{key}.parquet"
if path.exists():
    print("→ Lendo do cache:", path)
    return pq.read_table(path).to_pandas()
print("→ Calculando...")
merged = pd.merge_asof(
    left_df.sort_index(), right_df.sort_index(),
    left_index=True, right_index=True,
    direction="backward", tolerance=pd.to_timedelta(tolerance)
)
merged["ratio"] = merged.iloc[:,0] / merged.iloc[:,1]
pq.write_table(pa.Table.from_pandas(merged), path, compression="zstd")
merged


In [ ]:
# Função de hash estável

# Função que calcula e salva em cache
def compute_with_cache(left_df, right_df, cache_dir=Path("cache"), tolerance="1D"):
    cache_dir.mkdir(exist_ok=True)
    key = hashlib.sha256(
        (hash_dataframe(left_df) + hash_dataframe(right_df) + tolerance).encode()
    ).hexdigest()
    path = cache_dir / f"{key}.parquet"
    if path.exists():
        print("→ Lendo do cache:", path)
        return pq.read_table(path).to_pandas()
    print("→ Calculando...")
    merged = pd.merge_asof(
        left_df.sort_index(), right_df.sort_index(),
        left_index=True, right_index=True,
        direction="backward", tolerance=pd.to_timedelta(tolerance)
    )
    merged["ratio"] = merged.iloc[:,0] / merged.iloc[:,1]
    pq.write_table(pa.Table.from_pandas(merged), path, compression="zstd")
    return merged

# Executar duas vezes
df1 = compute_with_cache(quotes, indicators)
df2 = compute_with_cache(quotes, indicators)


### Métodos FLY

In [ ]:
import datetime

import pandas as pd

In [ ]:
raw_quotes = pd.read_csv("quotes_ALPA3.csv")
raw_statements = pd.read_csv("statements_con.csv")
raw_indicators = pd.read_csv("indicators_11.csv")

In [ ]:
def _treat_quotes(q, cutoff=None):
    if not cutoff:
        cutoff = datetime.datetime(year=2010, month=12, day=31)

    q["date"] = pd.to_datetime(q["date"])
    q = q.sort_values("date").drop_duplicates(subset=["date"]).set_index("date")
    if cutoff:
        q = q[q.index >= cutoff]
    return q

In [ ]:
def _treat_statements(s, c):
    if c.empty:
        return pd.DataFrame()

    key_columns = ["company_name", "quarter", "grupo", "quadro", "account"]
    s["account_description"] = s["account"].str.cat(s["description"], sep=" - ")
    s["quarter"] = pd.to_datetime(s["quarter"])
    s = s.sort_values(key_columns)
    s = s.drop_duplicates(subset=key_columns, keep="last")
    statements_wide = s.pivot_table(index="quarter",
                            columns="account_description",  # use "description" se preferir nomes ou "account" se preferir contas
                            values="value",
                            aggfunc="last").sort_index()
    statements_wide.columns.name = None

    s = statements_wide.sort_index().reindex(c.index, method="ffill")
    if s.iloc[0].isna().any():
        s = s.bfill()

    return s


In [ ]:
def _indicators(i, c):
    if c.empty:
        return pd.DataFrame()

    indicators_wide = indicators.copy()
    indicators_wide["date"] = pd.to_datetime(indicators_wide["date"])
    indicators_wide = indicators_wide.sort_values("date").drop_duplicates(subset=["date"]).set_index("date")

    i = indicators_wide.sort_index().reindex(c.index, method="ffill")
    if i.iloc[0].isna().any():
        i = i.bfill()

    return i

In [ ]:
cutoff = datetime.datetime(year=2024, month=12, day=31)
cutoff = raw_statements['quarter'].min()

q = _quotes(raw_quotes, cutoff)
s = _statements(raw_statements, q)
i = _indicators(raw_indicators, q)


In [ ]:
s

## GitIngest

In [1]:
import nest_asyncio
from gitingest import ingest # Use a versão SÍNCRONA

# 1. Aplica o patch no loop de eventos do Jupyter.
# O patch permite que a função ingest (que usa asyncio.run) execute 
# dentro do loop já ativo do Jupyter, evitando o RuntimeError.
nest_asyncio.apply() 

repo = "https://github.com/faustostangler/FLY"

# 2. Chame a função SÍNCRONA. 
# Se a falha do subprocesso persistir com a versão async, esta é a última tentativa.
summary, tree, content = ingest(repo) # Não use 'await'

print("Sucesso. O repositório foi ingerido.")
print("\nPrimeiros 500 caracteres do Tree:")
print(tree[:500])

2025-11-19 09:15:13.688 | INFO     | gitingest.entrypoint:ingest_async:89 | Starting ingestion process | {"source":"https://github.com/faustostangler/FLY"}
2025-11-19 09:15:13.688 | INFO     | gitingest.entrypoint:ingest_async:98 | Parsing remote repository | {"source":"https://github.com/faustostangler/FLY"}


NotImplementedError: 

2025-11-19 09:15:14.610 | INFO     | ipykernel.ipkernel:do_execute:489 | Exception in execute request:
---------------------------------------------------------------------------
NotImplementedError                       Traceback (most recent call last)
Cell In[1], line 13
      9 repo = "https://github.com/faustostangler/FLY"
     11 # 2. Chame a função SÍNCRONA. 
     12 # Se a falha do subprocesso persistir com a versão async, esta é a última tentativa.
---> 13 summary, tree, content = ingest(repo) # Não use 'await'
     15 print("Sucesso. O repositório foi ingerido.")
     16 print("\nPrimeiros 500 caracteres do Tree:")

File d:\Fausto Stangler\Documentos\Python\FLY\.venv\lib\site-packages\gitingest\entrypoint.py:209, in ingest(source, max_file_size, include_patterns, exclude_patterns, branch, tag, include_gitignored, include_submodules, token, output)
    151 def ingest(
    152     source: str,
    153     *,
   (...)
    162     output: str | None = None,
    163 ) -> tuple[str

## AI

In [ ]:
with open(r"D:\ollama\models\blobs\sha256-64e8f4d6856fca67b11f0875f9552264e62ababbe68fd6ddd0129015ee6df70a", "rb") as f:
    header = f.read(100_000)   # só os primeiros 100 KB

print(header[:500])  # mostra versão GGUF, magic number, etc.

b'GGUF\x03\x00\x00\x00\xf5\x00\x00\x00\x00\x00\x00\x00$\x00\x00\x00\x00\x00\x00\x00\x14\x00\x00\x00\x00\x00\x00\x00general.architecture\x08\x00\x00\x00\x04\x00\x00\x00\x00\x00\x00\x00phi3\x0c\x00\x00\x00\x00\x00\x00\x00general.type\x08\x00\x00\x00\x05\x00\x00\x00\x00\x00\x00\x00model\x0c\x00\x00\x00\x00\x00\x00\x00general.name\x08\x00\x00\x00\x1a\x00\x00\x00\x00\x00\x00\x00Phi 3 Medium 128k Instruct\x10\x00\x00\x00\x00\x00\x00\x00general.finetune\x08\x00\x00\x00\r\x00\x00\x00\x00\x00\x00\x00128k-instruct\x10\x00\x00\x00\x00\x00\x00\x00general.basename\x08\x00\x00\x00\x05\x00\x00\x00\x00\x00\x00\x00Phi-3\x12\x00\x00\x00\x00\x00\x00\x00general.size_label\x08\x00\x00\x00\x06\x00\x00\x00\x00\x00\x00\x00medium\x0f\x00\x00\x00\x00\x00\x00\x00general.license\x08\x00\x00\x00\x03\x00\x00\x00\x00\x00\x00\x00mit\x14\x00\x00\x00\x00\x00\x00\x00general.license.link\x08\x00\x00\x00P\x00\x00\x00\x00\x00\x00\x00https://huggingface.co/microsoft/Phi-3-medium-128k-instruct/resolve/main/LICENSE\x0c\x00\x0

In [ ]:
gguf_path = r"D:\ollama\models\blobs\sha256-ff1d1fc78170d787ee1201778e2dd65ea211654ca5fb7d69b5a2e7b123a50373"

with open(gguf_path, "rb") as f:
    header = f.read(100_000)   # só os primeiros 100 KB

print(header[:500])  # mostra versão GGUF, magic number, etc.

b"GGUF\x03\x00\x00\x00\xd0\x01\x00\x00\x00\x00\x00\x00\x1d\x00\x00\x00\x00\x00\x00\x00\x14\x00\x00\x00\x00\x00\x00\x00general.architecture\x08\x00\x00\x00\x06\x00\x00\x00\x00\x00\x00\x00gemma2\x0c\x00\x00\x00\x00\x00\x00\x00general.name\x08\x00\x00\x00\r\x00\x00\x00\x00\x00\x00\x00gemma-2-9b-it\x15\x00\x00\x00\x00\x00\x00\x00gemma2.context_length\x04\x00\x00\x00\x00 \x00\x00\x17\x00\x00\x00\x00\x00\x00\x00gemma2.embedding_length\x04\x00\x00\x00\x00\x0e\x00\x00\x12\x00\x00\x00\x00\x00\x00\x00gemma2.block_count\x04\x00\x00\x00*\x00\x00\x00\x1a\x00\x00\x00\x00\x00\x00\x00gemma2.feed_forward_length\x04\x00\x00\x00\x008\x00\x00\x1b\x00\x00\x00\x00\x00\x00\x00gemma2.attention.head_count\x04\x00\x00\x00\x10\x00\x00\x00\x1e\x00\x00\x00\x00\x00\x00\x00gemma2.attention.head_count_kv\x04\x00\x00\x00\x08\x00\x00\x00'\x00\x00\x00\x00\x00\x00\x00gemma2.attention.layer_norm_rms_epsilon\x06\x00\x00\x00\xbd7\x865\x1b\x00\x00\x00\x00\x00\x00\x00gemma2.attention.key_length\x04\x00\x00\x00\x00\x01\x00\x00

In [ ]:
from gguf import GGUFReader

gguf_path = r"D:\ollama\models\blobs\sha256-64e8f4d6856fca67b11f0875f9552264e62ababbe68fd6ddd0129015ee6df70a"
gguf_path = r"D:\ollama\models\blobs\sha256-ff1d1fc78170d787ee1201778e2dd65ea211654ca5fb7d69b5a2e7b123a50373"

# Só lê metadados (não carrega os 5 GB na RAM)
reader = GGUFReader(gguf_path)

def get_field(name):
    field = reader.fields.get(name)
    if not field:
        return None
    value = field.data[0]
    if isinstance(value, bytes):
        return value.decode('utf-8', errors='ignore').rstrip('\x00')
    return value

print("MODELO IDENTIFICADO".center(80, "="))
print(f"Nome          : {get_field('general.name')}")
print(f"Arquitetura   : {get_field('general.architecture')}")
print(f"Quantização   : mostly Q4_0 (file_type = {get_field('general.file_type')})")
print(f"Tamanho       : {sum(t.n_bytes for t in reader.tensors)/1024**3:.3f} GB")
print(f"Tensores      : {len(reader.tensors)}")
print(f"Context length: {get_field('llama.context_length') or 131072}")
print(f"Camadas       : {get_field('llama.block_count')}")
print(f"Hidden size   : {get_field('llama.embedding_length')}")
print()

# ================== VISUALIZAR os tensores (só metadados) ==================
print("PRIMEIROS 15 TENSORES".center(80, "-"))
for i, tensor in enumerate(reader.tensors[:15]):
    size_mb = tensor.n_bytes / 1024 / 1024
    print(f"{i+1:2d}. {tensor.name:60} {str(list(tensor.shape)):18} {tensor.tensor_type.name:10} {size_mb:6.1f} MB")

print("\nÚLTIMOS 5 TENSORES".center(80, "-"))
for tensor in reader.tensors[-5:]:
    size_mb = tensor.n_bytes / 1024 / 1024
    print(f"    {tensor.name:60} {str(list(tensor.shape)):18} {tensor.tensor_type.name:10} {size_mb:6.1f} MB")

# ================== QUER VER O CONTEÚDO REAL DE UM TENSOR PEQUENO? ==================
# Exemplo: o tensor de bias do LayerNorm da saída (só 4096 floats → 16 KB)
print("\nCONTEÚDO REAL DO TENSOR 'output_norm.weight' (primeiros 50 valores):".center(80, "="))
for tensor in reader.tensors:
    if tensor.name == "output_norm.weight":
        # Agora sim carregamos só esse tensor na memória
        tensor_data = tensor.data  # ← aqui ele finalmente lê os bytes desse tensor
        print(tensor_data[:50].astype('float32')) 

==============================MODELO IDENTIFICADO===============================
Nome          : 4
Arquitetura   : 4
Quantização   : mostly Q4_0 (file_type = 3)
Tamanho       : 5.064 GB
Tensores      : 464
Context length: 131072
Camadas       : None
Hidden size   : None

-----------------------------PRIMEIROS 15 TENSORES------------------------------
 1. token_embd.weight                                            [np.uint64(3584), np.uint64(256000)] Q6_K        717.8 MB
 2. blk.0.attn_norm.weight                                       [np.uint64(3584)]  F32           0.0 MB
 3. blk.0.ffn_down.weight                                        [np.uint64(14336), np.uint64(3584)] Q4_0         27.6 MB
 4. blk.0.ffn_gate.weight                                        [np.uint64(3584), np.uint64(14336)] Q4_0         27.6 MB
 5. blk.0.ffn_up.weight                                          [np.uint64(3584), np.uint64(14336)] Q4_0         27.6 MB
 6. blk.0.post_attention_norm.weight                 

In [35]:
# ================================
# pip install tiktoken numpy gguf pandas
# ================================
import tiktoken
import numpy as np
from gguf import GGUFReader
import pandas as pd
import os
import re

gguf_path = r"D:\ollama\models\blobs\sha256-64e8f4d6856fca67b11f0875f9552264e62ababbe68fd6ddd0129015ee6df70a"
encoding = tiktoken.get_encoding("o200k_base")
reader = GGUFReader(gguf_path)

# Carrega o tensor de embedding UMA VEZ (717 MB na RAM)
print("Carregando token_embd.weight (uma única vez)...")
embedding_matrix = None
for tensor in reader.tensors:
    if tensor.name == "token_embd.weight":
        embedding_matrix = tensor.data.astype(np.float32)
        print(f"Embedding carregado: {embedding_matrix.shape} → {embedding_matrix.nbytes/1024**3:.2f} GB")
        break
if embedding_matrix is None:
    raise ValueError("Tensor token_embd.weight não encontrado!")

def salvar_vetores(frase: str, pasta_saida: str = "files"):
    os.makedirs(pasta_saida, exist_ok=True)
    
    # Sanitiza nome do arquivo (remove caracteres inválidos)
    nome_arquivo_base = re.sub(r'[<>:"/\\|?*\x00-\x1f]', '_', frase.strip())
    if not nome_arquivo_base:
        nome_arquivo_base = "frase_vazia"
    
    tokens = encoding.encode(frase)
    token_strs = [encoding.decode([t]).replace("\n", "\\n").replace("\t", "\\t") for t in tokens]
    
    print(f"\nProcessando: {frase!r}")
    print(f"Tokens: {token_strs} | IDs: {tokens}")
    
    vetores = []
    for token_str, token_id in zip(token_strs, tokens):
        if token_id >= len(embedding_matrix):
            print(f"Token fora do vocabulário: {token_str} (ID {token_id})")
            continue
        vetor = embedding_matrix[token_id]
        vetores.append(vetor)
        
        # Salva CSV individual por token
        df_token = pd.DataFrame([vetor], index=[f"{token_str}_id{token_id}"])
        nome_token = f"{nome_arquivo_base}__{token_str}_id{token_id}.csv"
        nome_token = re.sub(r'[<>:"/\\|?*\x00-\x1f]', '_', nome_token)[:200]
        df_token.to_csv(os.path.join(pasta_saida, nome_token), header=False)
    
    if vetores:
        # Salva CSV com todos os vetores (um por linha)
        df_todos = pd.DataFrame(vetores, index=[f"token_{i}_{s}" for i, s in enumerate(token_strs)])
        df_todos.to_csv(os.path.join(pasta_saida, f"{nome_arquivo_base}__todos_os_tokens.csv"))
        
        # Salva CSV com a SOMA dos vetores (o que o modelo "vê" da frase inteira)
        vetor_soma = np.sum(vetores, axis=0)
        df_soma = pd.DataFrame([vetor_soma], index=[f"SOMA_{nome_arquivo_base}"])
        df_soma.to_csv(os.path.join(pasta_saida, f"{nome_arquivo_base}__SOMA.csv"))
        
        print(f"Salvo em '{pasta_saida}/':")
        print(f"   → {len(vetores)} CSVs individuais")
        print(f"   → {nome_arquivo_base}__todos_os_tokens.csv")
        print(f"   → {nome_arquivo_base}__SOMA.csv  ← este é o vetor final da frase!")

# Rode isso com o código anterior
salvar_vetores("rei")
salvar_vetores("homem")
salvar_vetores("mulher")
salvar_vetores("rainha")

Carregando token_embd.weight (uma única vez)...
Embedding carregado: (32064, 2880) → 0.34 GB

Processando: 'rei'
Tokens: ['rei'] | IDs: [13409]
Salvo em 'files/':
   → 1 CSVs individuais
   → rei__todos_os_tokens.csv
   → rei__SOMA.csv  ← este é o vetor final da frase!

Processando: 'homem'
Tokens: ['hom', 'em'] | IDs: [26848, 347]
Salvo em 'files/':
   → 2 CSVs individuais
   → homem__todos_os_tokens.csv
   → homem__SOMA.csv  ← este é o vetor final da frase!

Processando: 'mulher'
Tokens: ['mul', 'her'] | IDs: [44857, 999]
Token fora do vocabulário: mul (ID 44857)
Salvo em 'files/':
   → 1 CSVs individuais
   → mulher__todos_os_tokens.csv
   → mulher__SOMA.csv  ← este é o vetor final da frase!

Processando: 'rainha'
Tokens: ['rain', 'ha'] | IDs: [39775, 1716]
Token fora do vocabulário: rain (ID 39775)
Salvo em 'files/':
   → 1 CSVs individuais
   → rainha__todos_os_tokens.csv
   → rainha__SOMA.csv  ← este é o vetor final da frase!


In [45]:
rei      = pd.read_csv("files/rei__SOMA.csv").iloc[0, 1:].values
homem    = pd.read_csv("files/homem__SOMA.csv").iloc[0, 1:].values
mulher   = pd.read_csv("files/mulher__SOMA.csv").iloc[0, 1:].values
rainha   = pd.read_csv("files/rainha__SOMA.csv").iloc[0, 1:].values

resultado = rei - homem + mulher

# Qual palavra tem o vetor mais parecido com "resultado"?
from sklearn.metrics.pairwise import cosine_similarity
similaridade_com_rainha = cosine_similarity([resultado], [rainha])[0][0]
# → geralmente > 0.65 (muito alto!)
similaridade_com_rainha

np.float64(-0.006921963155809647)

In [51]:
# ================================
# BUSCA SEMÂNTICA PERFEITA PARA PHI-3 MEDIUM 128K (14B)
# ================================
import tiktoken
import numpy as np
from gguf import GGUFReader
import os

# Seu modelo Phi-3 Medium 128k
gguf_path = r"D:\ollama\models\blobs\sha256-ff1d1fc78170d787ee1201778e2dd65ea211654ca5fb7d69b5a2e7b123a50373"

print("Carregando modelo Phi-3 Medium 128k...")
reader = GGUFReader(gguf_path)

# === Detecta automaticamente o tensor de embedding ===
embedding_tensor = None
for tensor in reader.tensors:
    if tensor.name == "token_embd.weight":
        embedding_tensor = tensor
        vocab_size = tensor.shape[0]
        dim = tensor.shape[1]
        print(f"Phi-3 detectado: {vocab_size:,} tokens × {dim} dim | {tensor.tensor_type.name}")
        break

# === Tokenizer correto do Phi-3 (cl100k_base) ===
enc = tiktoken.get_encoding("cl100k_base")   # ← É esse mesmo! Funciona 100% com Phi-3
print(f"Tokenizer carregado: {enc.name}")

# =================================================================
# BUSCA SEMÂNTICA LEVE E RÁPIDA (nunca explode RAM)
# =================================================================
def buscar_semantica(texto: str, top_k: int = 15):
    ids = enc.encode(texto)
    ids = [i for i in ids if i < vocab_size]  # proteção
    if not ids:
        print("Nenhum token válido encontrado!")
        return

    # Desquantiza só os tokens da frase (máximo 50 vetores → ~500 KB)
    vetores = [np.array(embedding_tensor.data[i], dtype=np.float32) for i in ids]
    vetor_query = np.mean(vetores, axis=0)

    # Calcula similaridade em blocos (nunca carrega tudo na RAM)
    scores = np.zeros(vocab_size, dtype=np.float32)
    batch_size = 2048
    query_norm = np.linalg.norm(vetor_query)

    for start in range(0, vocab_size, batch_size):
        end = min(start + batch_size, vocab_size)
        batch = np.array(embedding_tensor.data[start:end], dtype=np.float32)
        batch_norms = np.linalg.norm(batch, axis=1)
        dots = batch @ vetor_query
        cosines = dots / (batch_norms * query_norm + 1e-8)
        scores[start:end] = cosines

    melhores = np.argsort(scores)[::-1][:top_k]

    print(f"\nBuscando → \"{texto}\" ({len(ids)} tokens)")
    print("-" * 75)
    for i, idx in enumerate(melhores):
        token_str = enc.decode([idx]).replace("\n", "\\n")
        print(f"{i+1:2}. {token_str:30} (id {idx:6}) → {scores[idx]:.4f}")

# =================================================================
# TESTE AGORA! (vai funcionar perfeitamente)
# =================================================================
buscar_semantica("mulher poderosa")
buscar_semantica("inteligência artificial")
buscar_semantica("gato")
buscar_semantica("rei")
buscar_semantica("amor")
buscar_semantica("felicidade")
buscar_semantica("carro vermelho rápido")
buscar_semantica("rei - homem + mulher")   # analogia clássica!

Carregando modelo Phi-3 Medium 128k...
Phi-3 detectado: 3,584 tokens × 256000 dim | Q6_K
Tokenizer carregado: cl100k_base

Buscando → "mulher poderosa" (1 tokens)
---------------------------------------------------------------------------
 1. her                            (id   1964) → 1.0000
 2.                               (id    211) → 0.7724
 3. cre                            (id    846) → 0.7723
 4. std                            (id   1872) → 0.7703
 5.                               (id    194) → 0.7668
 6. sl                             (id   3306) → 0.7665
 7. ST                             (id    790) → 0.7662
 8. ym                             (id   1631) → 0.7653
 9. Table                          (id   2620) → 0.7649
10. ether                          (id   2791) → 0.7648
11. Request                        (id   1939) → 0.7644
12. its                            (id   1220) → 0.7642
13. ank                            (id   1201) → 0.7641
14. _st                          

In [52]:
token = "amor"
# pip install llama-cpp-python
from llama_cpp import Llama

llm = Llama(
    model_path=r"D:\ollama\models\blobs\sha256-ff1d1fc78170d787ee1201778e2dd65ea211654ca5fb7d69b5a2e7b123a50373",
    n_ctx=128,
    verbose=False
)

# Dá o token "amor" e pede o próximo
output = llm(
    token,
    max_tokens=1,
    echo=False,
    logprobs=20  # ← mostra as 20 maiores probabilidades!
)

print("Próximos tokens mais prováveis depois de 'amor':")
for item in output['choices'][0]['logprobs']['top_logprobs'][0].items():
    token_str = llm.detokenize([int(item[0])]).decode('utf-8', errors='ignore')
    print(f"  {token_str:15} → prob: {np.exp(item[1]):.5%}")

ModuleNotFoundError: No module named 'llama_cpp'

In [56]:
import requests
import json

def proximos_tokens(texto: str, top_k: int = 20):
    url = "http://localhost:11434/api/chat"
    
    payload = {
        "model": "llama3.2:3b",        # ← mude se o seu modelo tem outro nome
        "messages": [{"role": "user", "content": texto}],
        "stream": False,
        "options": {"temperature": 0},  # temperatura 0 = mais determinístico
        "logprobs": top_k               # ← aqui pede os top_k logprobs
    }
    
    r = requests.post(url, json=payload)
    data = r.json()
    
    # Extrai os top logprobs
    try:
        top_logprobs = data["message"]["logprobs"]["top_logprobs"][0]
    except KeyError:
        print("Erro: logprobs não retornado. Verifique se o modelo suporta.")
        print("Resposta completa:", data)
        return
    
    print(f"\nPróximos tokens mais prováveis depois de: \"{texto}\"")
    print("-" * 65)
    for token_id_str, logprob in top_logprobs.items():
        token_id = int(token_id_str)
        prob = 2 ** logprob  # Ollama usa log2
        # Decodifica o token usando o próprio Ollama (jeito mais simples)
        try:
            token_text = requests.post("http://localhost:11434/api/detokenize", 
                json={"tokens": [token_id]}).json()["tokens"][0]
        except:
            token_text = f"<{token_id}>"
        print(f"{token_text:20} → {prob:.4%}  (logprob: {logprob:.3f})")

# TESTA AGORA!
proximos_tokens("amor")


Erro: logprobs não retornado. Verifique se o modelo suporta.
Resposta completa: {'error': 'json: cannot unmarshal number into Go struct field ChatRequest.logprobs of type bool'}


In [58]:
import requests
import json

def tentar_logprobs(texto: str, top_k: int = 20):
    url = "http://localhost:11434/v1/chat/completions"  # Endpoint OpenAI-compatível
    
    payload = {
        "model": "llama3.1",
        "messages": [{"role": "user", "content": texto}],
        "stream": False,
        "temperature": 0,
        "logprobs": True,    # Tenta como booleano primeiro
        "top_logprobs": top_k  # Número de tokens (pode falhar)
    }
    
    try:
        r = requests.post(url, json=payload)
        data = r.json()
        print("Resposta completa:", json.dumps(data, indent=2))  # Debug: veja o que veio
    except Exception as e:
        print(f"Erro: {e}")
        # Fallback: geração sem logprobs
        print("Fallback: Gerando sem logprobs...")
        payload_simple = {k: v for k, v in payload.items() if k not in ["logprobs", "top_logprobs"]}
        r_simple = requests.post(url, json=payload_simple)
        print("Resposta simples:", r_simple.json()["choices"][0]["message"]["content"])

# Teste
tentar_logprobs("amor")

Resposta completa: {
  "error": {
    "message": "llama runner process has terminated: error loading model: unable to allocate CUDA0 buffer",
    "type": "api_error",
    "param": null,
    "code": null
  }
}


In [2]:
import tiktoken
from transformers import AutoTokenizer
import re

class TokenCounter:
    @staticmethod
    def count_gpt(text: str) -> dict:
        """Para GPT-4o, GPT-4, Claude, Gemini, Grok → todos usam cl100k_base"""
        enc = tiktoken.get_encoding("cl100k_base")
        tokens = enc.encode(text)
        return {
            "model": "GPT-4o / Claude / Grok / Gemini",
            "tokens": len(tokens),
            "tokens_list": [enc.decode([t]) for t in tokens],
            "tokens_ids": tokens
        }

    @staticmethod
    def count_llama3(text: str) -> dict:
        """Llama 3 / Llama 3.1 / Llama 3.2"""
        tokenizer = AutoTokenizer.from_pretrained("meta-llama/Meta-Llama-3.1-8B")
        tokens = tokenizer.encode(text)
        return {
            "model": "Llama 3 / 3.1 / 3.2",
            "tokens": len(tokens),
            "tokens_list": [tokenizer.decode([t]) for t in tokens],
            "tokens_ids": tokens
        }

    @staticmethod
    def show_tokens(text: str, model: str = "gpt4"):
        """Mostra visualmente cada token (MELHOR JEITO DE ENTENDER)"""
        if model.lower() in ["gpt", "gpt4", "gpt-4o", "claude", "grok", "gemini"]:
            enc = tiktoken.get_encoding("cl100k_base")
        elif "llama" in model.lower():
            enc = AutoTokenizer.from_pretrained("meta-llama/Meta-Llama-3.1-8B")
            tokens = enc.encode(text)
            return [enc.decode([t]).replace(" ", "␣").replace("\n", "⏎") for t in tokens]
        else:
            raise ValueError("Modelo não suportado")

        tokens = enc.encode(text)
        visual = []
        for t in tokens:
            token_str = enc.decode([t])
            # Mostra espaço como ␣ e quebra de linha como ⏎
            token_str = token_str.replace(" ", "␣").replace("\n", "⏎")
            visual.append(token_str)
        return visual

None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [7]:
texto = "Paciente refere ardência ocular há 3 dias. Usa Moura Brasil."

print("=== GPT-4o / Grok / Claude ===")
print(TokenCounter.count_gpt(texto), ["tokens"])  # → 18 tokens

print("\nTokens visualizados:")
print("".join(TokenCounter.show_tokens(texto, "gpt4")))

=== GPT-4o / Grok / Claude ===
{'model': 'GPT-4o / Claude / Grok / Gemini', 'tokens': 17, 'tokens_list': ['P', 'aciente', ' refere', ' ard', 'ência', ' oc', 'ular', ' há', ' ', '3', ' dias', '.', ' Usa', ' Mour', 'a', ' Brasil', '.'], 'tokens_ids': [47, 59303, 40008, 85009, 24625, 18274, 1299, 43222, 220, 18, 41470, 13, 97038, 51648, 64, 43025, 13]} ['tokens']

Tokens visualizados:
Paciente␣refere␣ardência␣ocular␣há␣3␣dias.␣Usa␣Moura␣Brasil.
